# 🌿 Setup Evidently project 

In [21]:
from evidently.ui.workspace import CloudWorkspace
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file
import os
try:
    ws = CloudWorkspace(token=os.getenv("EVIDENTLY_API_KEY"), url="https://app.evidently.cloud")

    project = ws.create_project("Simple-chatbot", org_id="019c99af-5151-7d0a-898d-aa9e84dca8aa")
    project.description = "Experiments with evaluating a simple chatbot using LLM-based evaluation and Evidently AI for tracking results."

    print("Project setup succeeded")

except Exception as e:
    print(f"Project setup failed: {e}")



Project setup succeeded


# 🏃 Run basic evaluations

In [23]:
import os

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from evidently import BinaryClassification, DataDefinition, Dataset, Report
from evidently.core.datasets import DatasetColumn
from evidently.descriptors import (
    BERTScore,
    ColumnTest,
    CustomColumnDescriptor,
    ExactMatch,
    HuggingFace,
    IncludesWords,
    LLMEval,
    PIILLMEval,
    DeclineLLMEval,
    CorrectnessLLMEval,
    FaithfulnessLLMEval,
    SemanticSimilarity,
    SentenceCount,
    Sentiment,
    TestSummary,
    TextLength,
)
from evidently.llm.templates import (
    BinaryClassificationPromptTemplate,
    MulticlassClassificationPromptTemplate,
)
from evidently.metrics import *
from evidently.presets import ClassificationPreset, DataSummaryPreset, TextEvals, ValueStats
from evidently.sdk.models import PanelMetric
from evidently.sdk.panels import DashboardPanelPlot
from evidently.tests import eq, gte, lte

load_dotenv(override=True)

eval_data = pd.read_json("datasets/dummy.json")

definition = DataDefinition(text_columns=["question", "context", "answer", "reference_answer"])

eval_df = Dataset.from_pandas(eval_data, data_definition=definition, descriptors=[
        SentenceCount("answer", alias="Sentence_Count"),
        TextLength("answer", alias="Answer Length",
                            tests=[gte(100, alias="Answer is too long")]),
        Sentiment("answer", alias="Sentiment")])

eval_df.add_descriptors(descriptors=[
    SemanticSimilarity(columns=["answer", "reference_answer"], alias="Semantic Similarity"),
    BERTScore(columns=["answer", "reference_answer"], alias="BERTScore"),
])


eval_df.add_descriptors(descriptors=[
     SemanticSimilarity(columns=["answer", "context"], alias="Hallucination proxy"),
     SemanticSimilarity(columns=["answer", "question"], alias="Relevance proxy")
])


eval_df.add_descriptors(descriptors=[
     FaithfulnessLLMEval("answer", context="context", alias="Faithfulness"),
     TextLength("answer", alias="Length")
])

correctness_multiclass = MulticlassClassificationPromptTemplate(
    pre_messages=[("system", "You are a judge that evaluates the factual alignment of two chatbot answers.")],
    criteria="""You are given a new answer and a reference answer. Classify the new answer based on how it compares to the reference.
    ===
    Reference: {reference_answer} """,
    category_criteria={
        "fully_correct": "The answer matches the reference in all factual and semantic details.",
        "incomplete": "The answer is correct in what it says but leaves out details from the reference.",
        "adds_claims": "The answer does not contradict reference but introduces new claims not supported by the reference.",
        "contradictory": "The answer contradicts specific facts or meaning in the reference.",
    },
    uncertainty="unknown",
    include_reasoning=True,
    include_scores=False,
)

eval_df.add_descriptors(descriptors=[
    LLMEval(
        "answer",
        template=correctness_multiclass,
        additional_columns={"reference_answer": "reference_answer"},
        provider="openai",
        model="gpt-4o-mini",
        alias="Multi-class correctness",
    )
])

completeness = BinaryClassificationPromptTemplate(
    pre_messages=[("system", "You are an evaluator assessing whether a chatbot response is sufficiently complete and informative on its own.")],
    criteria = """A COMPLETE response should be a full sentence or paragraph, and be easy to understand on its own.
    For example: "Yes, you can issue additional credit card for a relative.", or longer.
    A TOO-SHORT response is overly brief or vague—for example, just a number or a simple yes/no—without additional context.
    For example: "Yes, you can."
    """,
    target_category="complete",
    non_target_category="too-short",
    uncertainty="unknown",
    include_reasoning=True
)

eval_df.add_descriptors(descriptors=[
    LLMEval("answer",
        template=completeness,
        provider="openai",
        model="gpt-4o-mini",
        alias="Answer completeness"
    )
])
     

report = Report([TextEvals()])
my_eval = report.run(eval_df)

print(eval_df.as_dataframe())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


                                            question  \
0  How far is the parking facility from the airport?   
1            Is there a free shuttle to the airport?   
2  What is the cancellation policy if I cancel 3 ...   
3        What are the available parking price tiers?   
4          What is the maximum vehicle size allowed?   
5         Can I book a parking spot on the same day?   
6  How do I enter the parking facility when I arr...   
7  Is covered parking available, and what are the...   
8           What are the facility's operating hours?   
9  What happens if I stay longer than my original...   

                                             context  \
0  Located just 1 kilometer walking distance from...   
1  We provide a free shuttle bus service running ...   
2  Cancel more than 48 hours before scheduled arr...   
3  Standard Uncovered Parking: Spaces #1-60, star...   
4  Maximum Length: 5.0 meters. Maximum Width: 2.0...   
5  Can I book same-day parking? Yes, subject to

## 📖 Publish basic evalutions dataset to Evidently Cloud

In [ ]:
ws.add_dataset(
    dataset = eval_df,
    name = "dummy_dataset",
    project_id = project.id,
    description = "First dummy dataset with expert labels on review quality")
     

UUID('019c99fd-5367-766b-a0d1-9cbc1b8415a3')

# 🧪 Experiment with classification prompts and classification metrics

In [32]:
# 1. Name the experiment
name = "naive_prompt"
# 2. Define LLM judge prompt template

load_dotenv(override=True)

eval_data = pd.read_json("datasets/dummy.json")

feedback_quality = BinaryClassificationPromptTemplate(
        pre_messages=[("system", "You are evaluating the quality of responses given by a parking assistant chatbot.")],
        criteria = """An answer is GOOD when it contains all the information needed and meets the facts.
        An answer is BAD when it is incomplete, inaccurate, or fails to address the user's query properly.
        """,
        target_category="bad",
        non_target_category="good",
        uncertainty="unknown",
        include_reasoning=True,
        )

# display(eval_data.head())

# 3. Apply the LLM judge
eval_dataset = Dataset.from_pandas(
    pd.DataFrame(eval_data),
    data_definition=definition,
    descriptors=[
        LLMEval("answer",
                template=feedback_quality,  # We can pass new prompt version
                provider="openai",          # We can change the provider
                model="gpt-4o-mini",        # We can change the model
                alias="llm_judged_quality")
    ]
)

# 4. Add TRUE/FALSE for judge alignment
eval_dataset.add_descriptors([
    ExactMatch(columns=["llm_judged_quality", "expert_label"], alias="judge_alignment")
])

## 🧑‍⚖️ Publish dataset with llm-judges to Evidently cloud

In [33]:
ws.add_dataset(
    dataset = eval_dataset,
    name = "llm_judged_dataset",
    project_id = project.id,
    description = "First dummy dataset with expert labels on review quality")
     

UUID('019c9a18-7df5-7c15-8ba2-302f31e75ac1')

## Run and publish classification report

In [ ]:
def run_classification_report(eval_dataset, name=None, cloud_ws=None, project_id=None):

    df = eval_dataset.as_dataframe()
    df_filtered = df[df["llm_judged_quality"] != "UNKNOWN"]

    # Set the classification Data Definition
    definition_class = DataDefinition(
        classification=[BinaryClassification(
            target="expert_label",
            prediction_labels="llm_judged_quality",
            pos_label="bad"
        )],
        categorical_columns=["expert_label", "llm_judged_quality"]
    )

    # Create a Dataset object
    eval_data = Dataset.from_pandas(df_filtered, data_definition=definition_class)

    # Build classification report
    report = Report([
        ClassificationPreset(),
        ValueStats("llm_judged_quality"),
        ValueStats("expert_label")
    ])

    # Apply tag(s)
    tags = [name] if name else []

    print(f"eval_data: {eval_data}")
    print(f"tags: {tags}")
    my_eval = report.run(eval_data, tags=tags)

    # Optional: upload to Evidently Cloud
    if cloud_ws and project_id:
        cloud_ws.add_run(project_id, my_eval, include_data=True)

    return my_eval


my_eval = run_classification_report(
    eval_dataset,
    name=name,
    cloud_ws=ws, #Optional
    project_id=project.id #Optional
)

SyntaxError: invalid syntax (1105434202.py, line 30)

# 🖼️ Display eval

In [ ]:
my_eval